# Silver — GTFS Routes (SCD Type 2)

Cleans and historises the GTFS route registry.

**Source:** `bronze_gtfs_routes` (GTFS static feed, batch)
**Target:** `silver_routes_scd` — one row per *version* of a route
**Quarantine:** `silver_routes_scd_quarantine` — rejected rows with their reasons

Routes change slowly (a line is renamed, recoloured, reclassified), so the table
keeps full history via `valid_from` / `valid_to` / `is_current`. Changes are
detected by comparing a hash of the tracked attributes, which also makes the
load idempotent. 

Re-running it doesn't produce new versions.

In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable
from datetime import datetime

dbutils.widgets.text("catalog","")
dbutils.widgets.text("schema","")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

bronze_routes     = f"{catalog}.{schema}.bronze_gtfs_routes"
silver_routes     = f"{catalog}.{schema}.silver_routes"
routes_quarantine = f"{catalog}.{schema}.silver_routes_quarantine"

# One timestamp for the whole run so valid_to of the old version exactly
# matches valid_from of the new one. No gaps and no overlaps.
batch_ts = datetime.now()

print(f"Source:     {bronze_routes}")
print(f"Target:     {silver_routes}")
print(f"Quarantine: {routes_quarantine}")
print(f"Batch timestamp: {batch_ts}")

In [0]:
bronze_r = spark.read.table(bronze_routes)

# Cleaning, normalisation and enrichment
cleaned_r = (bronze_r         
    # business keys as strings
    .withColumn("route_id",         F.trim(F.col("route_id").cast("string")))
    .withColumn("agency_id",        F.col("agency_id").cast("string"))
    .withColumn("route_short_name", F.trim(F.col("route_short_name").cast("string")))

    # ZTM does not publish these: normalise empty strings to NULL
    .withColumn("route_long_name",
        F.when(F.trim(F.col("route_long_name")) == "", None)
         .otherwise(F.trim(F.col("route_long_name"))))
    .withColumn("route_desc",
        F.when(F.trim(F.col("route_desc")) == "", None)
         .otherwise(F.trim(F.col("route_desc"))))
    
    # source uses lowercase hex: normalise so the format is consistent
    .withColumn("route_color",      F.upper(F.trim(F.col("route_color"))))
    .withColumn("route_text_color", F.upper(F.trim(F.col("route_text_color"))))
    .withColumn("route_type",       F.col("route_type").cast("int"))

    # GTFS codes the type as a number: translate it for downstream consumers
    .withColumn("route_type_name",
        F.when(F.col("route_type").isin(0, 900), "Tram")
         .when(F.col("route_type").isin(3, 700), "Bus")
         .otherwise("Other"))
)

# Deduplication: bronze is append-only, so reloading GTFS leaves several rows
# per route. Keep the most recently ingested one.
w_r = Window.partitionBy("route_id").orderBy(F.col("ingestion_timestamp").desc())
cleaned_r = (cleaned_r
    .withColumn("_rn", F.row_number().over(w_r))
    .filter(F.col("_rn") == 1)
    .drop("_rn"))

# Explicit, named quality rules; each condition describes a VIOLATION
rules_r = {
    "missing_route_id":   F.col("route_id").isNull() | (F.col("route_id") == ""),
    "missing_short_name": F.col("route_short_name").isNull() | (F.col("route_short_name") == ""),
    "unknown_route_type": ~F.col("route_type").isin(0, 3, 700, 900),
    "invalid_color":      F.col("route_color").isNotNull() & ~F.col("route_color").rlike("^[0-9A-F]{6}$"),
    "invalid_text_color": F.col("route_text_color").isNotNull() & ~F.col("route_text_color").rlike("^[0-9A-F]{6}$"),
}

# Build an array of the rules each row violates
# array_compact drops the NULLs left by rules that passed, so an empty array means the row is clean.
reasons_r = F.array_compact(F.array(*[
    F.when(cond, F.lit(name)) for name, cond in rules_r.items()
]))

evaluated_r = cleaned_r.withColumn("rejection_reasons", reasons_r)

valid_r    = evaluated_r.filter(F.size("rejection_reasons") == 0)
rejected_r = evaluated_r.filter(F.size("rejection_reasons") > 0)

print(f"Valid routes:    {valid_r.count():,}")
print(f"Rejected routes: {rejected_r.count():,}")

# Rejected rows go to quarantine with their reasons, never silently dropped
if rejected_r.count() > 0:
    (rejected_r
        .select(
            F.col("route_id"),
            F.to_json(F.struct([bronze_r[c] for c in bronze_r.columns])).alias("raw_record"),
            F.col("rejection_reasons"),
            F.col("source_file"),
            F.lit(batch_ts).cast("timestamp").alias("quarantined_at"))
        .write.format("delta").mode("append").saveAsTable(routes_quarantine))
    print("Rejected rows written to quarantine.")
    display(rejected_r.select("route_id", "route_short_name", "rejection_reasons").limit(10))

In [0]:
# Attributes tracked for change detection, a change in any of these closes the
# current version and opens a new one. Columns not listed here (lineage,
# timestamps) can change without creating history.
tracked_r = ["route_short_name", "route_long_name", "route_desc", "route_type",
             "route_type_name", "route_color", "route_text_color", "agency_id"]


# A single hash over the tracked attributes replaces comparing eight columns
# one by one and avoids NULL-comparison pitfalls. 
# COALESCE: concat_ws skips NULLs so different records could hash identically.
incoming_r = (valid_r
    .withColumn("attributes_hash", F.sha2(F.concat_ws("||",
        *[F.coalesce(F.col(c).cast("string"), F.lit("<NULL>")) for c in tracked_r]
    ), 256))
    .select("route_id", *tracked_r, "attributes_hash", "source_file")
)

silver_r_dt = DeltaTable.forName(spark, silver_routes)
current_r   = spark.read.table(silver_routes).filter("is_current = true")

# Changed rows appear twice in the staged source, because a
# single MERGE can only UPDATE or INSERT a given matched row, never both.
#   merge_key = NULL     -> NULL never equals anything, so no match -> INSERT the new version
#   merge_key = route_id -> matches the current row                 -> UPDATE closes it
changed_r = (incoming_r.alias("s")
    .join(current_r.alias("t"), "route_id")
    .where(F.col("s.attributes_hash") != F.col("t.attributes_hash"))
    .select("s.*"))

# Three cases handled by this MERGE:
# new route       -> no match                   -> INSERT first version
# changed route   -> match, attributes changed  -> UPDATE closes the old version and the merge_key=NULL copy INSERTs the new one
# unchanged route -> match, attributes identical -> nothing happens (this is what makes re-runs safe)
staged_r = (changed_r.withColumn("merge_key", F.lit(None).cast("string"))
    .unionByName(incoming_r.withColumn("merge_key", F.col("route_id"))))

insert_r = {c: f"s.{c}" for c in ["route_id", *tracked_r, "attributes_hash", "source_file"]}
insert_r.update({
    "valid_from":          F.lit(batch_ts).cast("timestamp"),
    "valid_to":            F.lit(None).cast("timestamp"),
    "is_current":          F.lit(True),
    "silver_ingestion_ts": F.lit(batch_ts).cast("timestamp"),
})

(silver_r_dt.alias("t")
    .merge(staged_r.alias("s"), "t.route_id = s.merge_key AND t.is_current = true")
    .whenMatchedUpdate(
        condition = "t.attributes_hash <> s.attributes_hash",
        set = {"valid_to": F.lit(batch_ts).cast("timestamp"), "is_current": F.lit(False)})
    .whenNotMatchedInsert(values = insert_r)
    .execute())


# In SCD2 the row count exceeds the number of routes. Each historical version is its own row. 
# "Distinct routes" should stay constant across runs, while
# "Historical" grows only when an attribute actually changes.
sr = spark.read.table(silver_routes)
print(f"Total rows:      {sr.count():,}")
print(f"Current:         {sr.filter('is_current = true').count():,}")
print(f"Historical:      {sr.filter('is_current = false').count():,}")
print(f"Distinct routes: {sr.select('route_id').distinct().count():,}")

In [0]:
# Simulating rebranding line 10 changes its colour to green
(spark.read.table(bronze_routes)
    .filter(F.col("route_id") == 10)
    .withColumn("route_color", F.lit("00aa55"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .write.format("delta").mode("append").saveAsTable(bronze_routes))

In [0]:
# History of a single route ordered in time. Valid_to of the closed version equals valid_from of the next one
display(spark.read.table(silver_routes)
    .filter(F.col("route_id") == "10")
    .select("route_id", "route_short_name", "route_type_name",
            "route_color", "valid_from", "valid_to", "is_current")
    .orderBy("valid_from"))